<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/DL_Objdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To download and unzip the datasets

In [2]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset


Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
shoplifting-video-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

replace ./local_colab_storage/normal/normal-1.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

Dataset path Extraction



In [ ]:
!ls /content/local_colab_storage/normal | wc -l

In [ ]:
!ls /content/local_colab_storage/shoplifting | wc -l

In [ ]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)


Frame extraction

In [ ]:
def process_video(video_path, max_frames=32, resize_dim=(224, 224)):
    """
    Opens a video, uniformly extracts a fixed number of frames,
    resizes them, and normalizes pixel values.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Handle empty or corrupted videos
    if total_frames <= 0:
        cap.release()
        return None

    # Calculate uniform intervals to pick frames across the whole video duration
    # This ensures a 5-second video and a 20-second video both yield exactly 'max_frames'
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    video_frames = []

    for frame_idx in frame_indices:
        # Set the reader to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            break

        # 1. Convert color from BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 2. Resize the frame (e.g., to 224x224)
        frame_resized = cv2.resize(frame, resize_dim)

        # 3. Normalize pixel values by dividing by 255.0 (converts 0-255 integers to 0.0-1.0 floats)
        frame_normalized = frame_resized / 255.0

        video_frames.append(frame_normalized)

    cap.release()

    # If the video didn't have enough readable frames, pad it or skip it
    if len(video_frames) < max_frames:
        return None

    # Convert list of frames into a single NumPy array
    # Output shape: (16, 224, 224, 3)
    return np.array(video_frames, dtype=np.float32)

# --- EXAMPLE USAGE ON ONE FILE ---
# (Replace with your actual unzipped path from !ls)
## sample_path = "./local_colab_storage/"


# 1. Define the video extensions you want to look for
video_extensions = (".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm")

# 2. Grab all matching video paths recursively
video_paths = list(paths.list_files(args["dataset"], validExts=video_extensions))
print("video_paths: ", video_paths)

data=[]
labels=[]

#if os.path.exists(video_paths):
for video_path in video_paths:

  label = video_path.split(os.path.sep)[-2]
  labels.append(label)

  processed_tensor = process_video(video_path, max_frames=32, resize_dim=(224, 224))
  if processed_tensor is not None:
    data.append(processed_tensor)

  print("Video Processed Successfully!")
  print(f"Final Tensor Shape: {processed_tensor.shape}") # Expecting (32, 224, 224, 3)
  print(f"Min pixel value: {processed_tensor.min()}, Max pixel value: {processed_tensor.max()}")





In [ ]:
##labels = np.array(labels)

In [ ]:
# perform one-hot encoding on the labels
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

In [ ]:
# Convert lists to final NumPy arrays
X = np.array(data, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f"Data loading complete!")
print(f"X shape (Videos, Frames, H, W, Channels): {X.shape}")
print(f"y shape (Labels): {y.shape}")


Split into Train and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

CNN + LSTM

The CNN (MobileNetV2): Looks at each individual frame and extracts spatial features (like a hand, a bag, or a shelf).
The LSTM: Looks at how those spatial features change over the 16 frames to understand the action (the movement of hiding an item).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Create a sequential layer for augmentation
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.1)
])

# 1. Base CNN to extract features from a single frame
# We use MobileNetV2 because it is lightweight and fast
base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)

base_cnn.trainable = False  # Freeze weights as we don't destroy pre-trained weights by new training data during backpropagation


# Flatten the CNN output to a vector
pooling_layer = layers.GlobalAveragePooling2D()(base_cnn.output)
feature_extractor = models.Model(inputs=base_cnn.input, outputs=pooling_layer)

# 2. Complete Video Model
video_input = layers.Input(shape=(32, 224, 224, 3))
# TimeDistributed applies the CNN to all frames individually
augmented_input = layers.TimeDistributed(aug)(video_input)
encoded_frames = layers.TimeDistributed(feature_extractor)(augmented_input)


# LSTM tracks the movement across the timeline
x = layers.LSTM(64, dropout=0.5)(encoded_frames)
x = layers.Dense(32, activation='relu')(x)

# Output layer: Sigmoid activation for binary classification (0 or 1)
output = layers.Dense(2, activation='sigmoid')(x)

model = models.Model(inputs=video_input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()


 Training Model by feeding processed arrays into the model

In [ ]:
## Run training for 30 to 50 epochs, but use an EarlyStopping callback.
#This tells Colab to keep training as long as the validation loss is improving, and automatically stop if it plateaus, saves time.

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


history = model.fit(
    X_train, y_train,
    validation_split=0.5, # Uses a slice of training data to check performance mid-training
    epochs=30,
    batch_size=8 # Small batch size keeps memory safe
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get raw probability predictions (numbers between 0.0 and 1.0)
predictions = model.predict(X_test)
print(predictions)


In [ ]:
print(y_test)

In [ ]:
# 2. Convert probabilities to binary choices: if > 0.5, it's Shoplifting (1), else Normal (0)
##binary_predictions = (predictions > 0.5).astype(int)

binary_predictions = np.argmax(predictions, axis=1)

# Convert y_test from 2 columns to 1 column
y_test_labels = np.argmax(y_test, axis=1)

print(binary_predictions)

In [ ]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_labels, binary_predictions))



In [ ]:
print("\n--- Classification Report ---")
print(classification_report(y_test_labels, binary_predictions, target_names=['Normal', 'Shoplifting']))